In [1]:

# override_data = [
#     ("Sample Track 1", "Tushit Desai", None, "Raag Charukeshi", None, None, None, None),
#     ("Sample Track 2", None, None, None, "C# Minor", None, None, None)
# ]
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import col, when
from delta.tables import DeltaTable

# 1. Define the exact schema 
schema = StructType([
    StructField("Title", StringType(), True),
    StructField("Override_Singer", StringType(), True),
    StructField("Override_Composer", StringType(), True),
    StructField("Override_Raag", StringType(), True),
    StructField("Override_Scale", StringType(), True),
    StructField("Override_Movie", StringType(), True),
    StructField("Override_Album", StringType(), True),
    StructField("Override_Poet", StringType(), True)
])

# 2. Read from CSV and enforce schema simultaneously
overrides_df = spark.read.option("header", "true").schema(schema).csv("Files/Bronze/Manual_Overrides.csv")

# 3. Convert empty strings to literal nulls so the coalesce function works later
for col_name in overrides_df.columns:
    overrides_df = overrides_df.withColumn(col_name, when(col(col_name) == "", None).otherwise(col(col_name)))

# 4. Upsert (MERGE) into the manual overrides Delta table
table_name = "manual_metadata_overrides"

if spark.catalog.tableExists(table_name):
    target_table = DeltaTable.forName(spark, table_name)
    target_table.alias("target").merge(
        overrides_df.alias("source"),
        "target.Title = source.Title"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print("Overrides table updated successfully.")
else:
    overrides_df.write.format("delta").mode("overwrite").saveAsTable(table_name)
    print("Overrides table created successfully.")

StatementMeta(, e1a87e8c-2a97-4942-93d6-f92662d6193d, 3, Finished, Available, Finished, False)

Overrides table updated successfully.


In [2]:
# Instantly release Spark compute resources to prevent pipeline capacity errors
spark.stop()

StatementMeta(, e1a87e8c-2a97-4942-93d6-f92662d6193d, 4, Finished, Available, Finished, False)